In [ ]:
import pandas as pd
import re
import torch.nn as nn

In [ ]:
df=pd.read_csv('/content/gk_qna_dataset.csv')

In [ ]:
df.head(10)

In [ ]:
df['question']

In [ ]:
def tokenize(text):
  #lower
  text=text.lower()

  #remove quotes

  text=re.sub(r"[\"']","",text)
  text = re.sub(r"[^a-z0-9\s]", " ", text)
  text = re.sub(r"\s+", " ", text).strip()
  token=text.split()




  return token

In [ ]:
print(tokenize(df['question'][0]))

In [ ]:
vocab={

       '<UNK>':0
}

In [ ]:
len(vocab)

In [ ]:
def build_vocan(row):

  token_question=tokenize(row['question'])
  token_answer=tokenize(row['answer'])

  marge_token=token_question+token_answer
  print(marge_token)


  for token in marge_token:
    if token not in vocab:
      vocab[token] = len(vocab)

In [ ]:
df.apply(build_vocan,axis=1)

In [ ]:
len(vocab)

In [ ]:
def text_to_indices(text,vocab):
  text_indices=[]


  for token in tokenize(text):
    if token in vocab:
      text_indices.append(vocab[token])

    else:
      text_indices.append('<UNK>')
  return text_indices

In [ ]:
text_to_indices("what is phitron",vocab)

In [ ]:
len(df)

In [ ]:
index=0

text_to_indices(df.iloc[0,0],vocab)

In [ ]:
import torch
from torch.utils.data import Dataset,DataLoader

In [ ]:
class QAdatset(Dataset):


  def __init__(self,df,vocab):
    self.df=df
    self.vocab=vocab



  def __len__(self):
    return self.df.shape[0]

  def __getitem__(self,index):
    numeric_question=text_to_indices(self.df.iloc[index][0],self.vocab)
    numeric_ans=text_to_indices(self.df.iloc[index]['answer'],self.vocab)


    return torch.tensor(numeric_question),torch.tensor(numeric_ans)

In [ ]:
dataset=QAdatset(df,vocab)


In [ ]:
dataset[0]

In [ ]:
dataloader=DataLoader(dataset,batch_size=1,shuffle=True)

In [ ]:
for ques,ans in dataloader:
  print(ques[0],ans[0])

In [ ]:
x=torch.tensor([[[1,2,3]]])
x.shape
y=x.squeeze(0)
y.shape

In [ ]:
class SimpleRnn(nn.Module):

  def __init__(self,vocab_size,hidden_size=64,embedding_dim=50):
    super().__init__()
    self.embedding=nn.Embedding(vocab_size,embedding_dim)
    self.rnn=nn.RNN(embedding_dim,hidden_size,batch_first=True)
    self.fc=nn.Linear(hidden_size,vocab_size)

  def forward(self,question):

    embedded=self.embedding(question)
    _,final=self.rnn(embedded)
    final=final.squeeze(0)
    final=self.fc(final)


    return final




In [ ]:
model=SimpleRnn(vocab_size=len(vocab))

In [ ]:
model

In [ ]:
y=torch.tensor([1,2,3])
print(y)
y.shape


In [ ]:
criterion=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(model.parameters(),lr=0.001)

In [ ]:
epochs=50

for epoch in range(epochs):
  total_loss=0


  for question,answer in dataloader:
    out=model(question)
    target=answer[0][0].unsqueeze(0)

    los=criterion(out,target)
    optimizer.zero_grad()


    los.backward()
    optimizer.step()
    total_loss=total_loss+los.item()


  print(f"epch{epoch+1}/{epochs}:{total_loss}")


In [ ]:
vocab.items()

In [ ]:
idx_to_word={idx:word for word,idx in vocab.items()}
idx_to_word

In [ ]:
def predict(question_text,model,vocab,idx_to_word):
  model.eval()

  with torch.no_grad():
    indices = text_to_indices(question_text,vocab)

    if not indices:
      return "<UNK>"

    x = torch.tensor(indices).unsqueeze(0)  #batch size, sequence legth

    pred = torch.argmax(model(x),dim=1).item()


  return idx_to_word.get(pred,"<UNK>")

In [ ]:
# Test
tests = [
    "What is the capital of France?",
    "What is H2O commonly called?",
    "Which planet is known as the red planet?",
    "Which bird cannot fly?",
    "What is the capital of Japan ?",

]
for q in tests:
    print(f"{q:45} → {predict(q, model, vocab, idx_to_word)}")